In [1]:
from dotenv import load_dotenv
import os
import sys

load_dotenv(dotenv_path='/Users/munizasiddiqui/easyf1/.env')
API_KEY = os.getenv("GEMINI_API_KEY")
sys.path.append('/Users/munizasiddiqui/easyf1')

print(f"API key loaded: {'Yes' if API_KEY else 'NO'}")

API key loaded: Yes


In [2]:
#imports + fastf1 setup
import fastf1
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

fastf1.Cache.enable_cache('/Users/munizasiddiqui/easyf1/cache')

#drivers name mapping 
DRIVERS_NAMES = {
    'HAM': 'Lewis Hamilton', 'VER': 'Max Verstappen', 'LEC': 'Charles Leclerc',
    'PER': 'Sergio Perez', 'RUS': 'George Russell', 'SAI': 'Carlos Sainz',
    'NOR': 'Lando Norris', 'GAS': 'Pierre Gasly', 'OCO': 'Esteban Ocon',
    'ALO': 'Fernando Alonso', 'STR': 'Lance Stroll', 'BOT': 'Valtteri Bottas',
    'ZHO': 'Zhou Guanyu', 'MAG': 'Kevin Magnussen', 'HUL': 'Nico Hulkenberg',
    'TSU': 'Yuki Tsunoda', 'DEV': 'Nyck de Vries', 'ALB': 'Alexander Albon',
    'SAR': 'Logan Sargeant', 'PIA': 'Oscar Piastri', 'RIC': 'Daniel Ricciardo',
    'LAW': 'Liam Lawson', 'SAR': 'Logan Sargeant',
}

print(f"Fast F1 ready ")
print(f"Version:{fastf1.__version__}")

Fast F1 ready 
Version:3.8.3


In [3]:

import fastf1
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

fastf1.Cache.enable_cache('/Users/munizasiddiqui/easyf1/cache')

DRIVER_NAMES = {
    'HAM': 'Lewis Hamilton', 'VER': 'Max Verstappen', 'LEC': 'Charles Leclerc',
    'PER': 'Sergio Perez', 'RUS': 'George Russell', 'SAI': 'Carlos Sainz',
    'NOR': 'Lando Norris', 'GAS': 'Pierre Gasly', 'OCO': 'Esteban Ocon',
    'ALO': 'Fernando Alonso', 'STR': 'Lance Stroll', 'BOT': 'Valtteri Bottas',
    'ZHO': 'Zhou Guanyu', 'MAG': 'Kevin Magnussen', 'HUL': 'Nico Hulkenberg',
    'TSU': 'Yuki Tsunoda', 'DEV': 'Nyck de Vries', 'ALB': 'Alexander Albon',
    'SAR': 'Logan Sargeant', 'PIA': 'Oscar Piastri', 'RIC': 'Daniel Ricciardo',
    'LAW': 'Liam Lawson',
}


#CHUNKING FUNCTION

#Tier-1a: Pit stop function
#one chunk per pit stop 
#only capturing the lap where driver actaully pitted, instead of one chunk per lap

def create_pitstop_chunks(laps_data, session):
    """
        create one chunk per pit stop in the race
    
        parameters:
        laps_data = DataFrame
            Full laps dataframe from fastf1
        season: fastf1 season object
            The loaded race session
    
        Returns:
        chunks = list of dicts
            Each dict has 'text' and 'metadataa'
        """
    chunks = []
    race_name = session.event['EventName']
    race_year = session.event['EventDate'].year
    circuit   = session.event['Location']
    total_laps = int(laps_data['LapNumber'].max())

    #finding all laps where pit stop happened 
    pit_laps = laps_data[laps_data['PitInTime'].notna()].copy()
    for _, lap in pit_laps.iterrows():
        driver    = lap['Driver']
        full_name = DRIVER_NAMES.get(driver, driver)
        team      = lap['Team']
        lap_num   = int(lap['LapNumber'])
        compound  = lap['Compound']
        tyre_age  = int(lap['TyreLife']) if not pd.isna(lap['TyreLife']) else 0
        position  = int(lap['Position']) if not pd.isna(lap['Position']) else 0
        stint     = int(lap['Stint']) if not pd.isna(lap['Stint']) else 0

        # the next lap after pitting shows the new compound
        driver_laps = laps_data[laps_data['Driver'] == driver].sort_values('LapNumber')
        next_laps   = driver_laps[driver_laps['LapNumber'] > lap_num]
        new_compound = next_laps.iloc[0]['Compound'] if len(next_laps) > 0 else "unknown"

        #checking track status, was it under safety car?
        track_status = str(lap['TrackStatus'])
        sc_active    = '4' in track_status
        vsc_active   = '6' in track_status
        if sc_active:
            condition = "under a Safety Car"
        elif vsc_active:
            condition = "under a Virtual Safety Car"
        else:
            condition = "under green flag conditions"

        #Building the chunk text
        text = (
            f"PIT STOP: Lap {lap_num} of {total_laps}, {race_year} {race_name} "
            f"at {circuit}. {full_name} ({driver}) racing for {team} made a pit "
            f"stop from position {position}. Switched from {tyre_age}-lap-old "
            f"{compound} tyres to {new_compound} tyres, {condition}. "
            f"This ended stint {stint}."
        )
        metadata = {
            'chunk_type': 'pit_stop', 'race': f"{race_year} {race_name}",
            'circuit': circuit, 'year': race_year, 'lap': lap_num,
            'driver': driver, 'team': str(team), 'compound_from': str(compound),
            'compound_to': str(new_compound), 'tyre_age': tyre_age,
            'position': position, 'sc_active': sc_active, 'vsc_active': vsc_active,
        }
        chunks.append({'text': text, 'metadata': metadata})
    return chunks

# Test on the 2023 British GP
session = fastf1.get_session(2023, 'British Grand Prix', 'R')
session.load()
laps = session.laps
pit_chunks = create_pitstop_chunks(laps, session)

print(f"Pit stop chunks created: {len(pit_chunks)}")
print()
print("Example:")
print(pit_chunks[0]['text'])

core           INFO 	Loading data for British Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '4', '44', '81', '63', '11', '14', '23', '16', '55', '2', '77', '27', '18', '24', '22', '21', '10', '20', '31']


Pit stop chunks created: 26

Example:
PIT STOP: Lap 33 of 52, 2023 British Grand Prix at Silverstone. Max Verstappen (VER) racing for Red Bull Racing made a pit stop from position 1. Switched from 33-lap-old MEDIUM tyres to SOFT tyres, under a Safety Car. This ended stint 1.


In [4]:
#Tier-1b: Safety car period chunks 
#One chunk per sc/vsc in the race (not per driver)
#These will descriibe the whole period as a stretegic window

def create_sc_period_chunks(laps_data, session):
    """
    creates one chunk per safety car or vsc period in the race

    """

    chunks = []
    race_name = session.event['EventName']
    race_year = session.event['EventDate'].year
    circuit   = session.event['Location']
    total_laps = int(laps_data['LapNumber'].max())

    #Getting once track status per lap
    lap_status = (laps_data[['LapNumber', 'TrackStatus']]
                  .drop_duplicates(subset = 'LapNumber')
                  .sort_values('LapNumber'))

    events = []
    current = None
    for _, row in lap_status.iterrows():
        lap = int(row['LapNumber'])
        status = str (row['TrackStatus'])
        is_sc    = '4' in status
        is_vsc  = '6' in status

        if (is_sc or is_vsc) and current is None:
            current ={'type': 'Safety Car' if is_sc else 'Virtual Safety Car',
                      'start': lap}

        elif not (is_sc or is_vsc) and current is not None:
            current ['end'] = lap - 1
            events.append(current)
            current = None

    if current is not None:
        current['end'] = int(lap_status['LapNumber'].max())
        events.append(current)

    #building for each chunk
    for e in events:
        duration = e['end'] - e['start'] + 1
        text = (
            f"{e['type'].upper()} PERIOD: {race_year} {race_name} at {circuit}. "
            f"A {e['type']} was deployed on lap {e['start']} and lasted until "
            f"lap {e['end']} ({duration} laps total, out of {total_laps}). "
            f"This created a strategic pit window where teams could change "
            f"tyres at reduced time cost."
        )
        metadata = {
            'chunk_type': 'sc_period',
            'race'      : f"{race_year} {race_name}",
            'circuit'   : circuit,
            'year'      : race_year,
            'lap'       : e['start'],
            'sc_type'   : e['type'],
            'duration'  : duration,
        }
        chunks.append({'text': text, 'metadata': metadata})

    return chunks

sc_chunks = create_sc_period_chunks(laps, session)
print(f"SC period chunks created: {len(sc_chunks)}")
for c in sc_chunks:
    print()
    print(c['text'])

SC period chunks created: 1

SAFETY CAR PERIOD: 2023 British Grand Prix at Silverstone. A Safety Car was deployed on lap 33 and lasted until lap 38 (6 laps total, out of 52). This created a strategic pit window where teams could change tyres at reduced time cost.


In [5]:
#Tier-1c: Stint Summary Chunks 
# One chunk per stint (continous run on one set of tyres) per driver

def create_stint_chunks(laps_data, session):
    chunks = []
    race_name = session.event['EventName']
    race_year = session.event['EventDate'].year
    circuit   = session.event['Location']
    total_laps = int(laps_data['LapNumber'].max())

    for driver in laps_data['Driver'].unique():
        driver_laps = laps_data[laps_data['Driver'] == driver]
        full_name = DRIVER_NAMES.get(driver, driver)
        team = driver_laps['Team'].iloc[0] if len(driver_laps) > 0 else 'Unknown'

        #looping through each stint driver ran 
        for stint_num in sorted(driver_laps['Stint'].dropna().unique()):
            stint_laps = driver_laps[driver_laps['Stint'] == stint_num]
            if len(stint_laps) == 0:
                continue

            compound = stint_laps['Compound'].iloc[0]
            first_lap = int(stint_laps['LapNumber'].min())
            last_lap = int(stint_laps['LapNumber'].max())
            stint_len = last_lap - first_lap + 1
            max_age = int(stint_laps['TyreLife'].max()) if not pd.isna(stint_laps['TyreLife'].max()) else 0

            text = (
                f"STINT: {race_year} {race_name} at {circuit}. "
                f"{full_name} ({driver}, {team}) ran stint {int(stint_num)} on "
                f"{compound} tyres from lap {first_lap} to lap {last_lap} "
                f"({stint_len} laps, tyres reached {max_age} laps of age)."
            )
            metadata = {
                'chunk_type': 'stint',
                'race'      : f"{race_year} {race_name}",
                'circuit'   : circuit,
                'year'      : race_year,
                'lap'       : first_lap,
                'driver'    : driver,
                'team'      : str(team),
                'compound'  : str(compound),
                'stint'     : int(stint_num),
                'stint_length': stint_len,
            }
            chunks.append({'text': text, 'metadata': metadata})

    return chunks

stint_chunks = create_stint_chunks(laps, session)
print(f"Stint chunks created: {len(stint_chunks)}")
print()
print("Example stint chunks (first 3):")
for c in stint_chunks[:3]:
    print(c['text'])
    print()



Stint chunks created: 44

Example stint chunks (first 3):
STINT: 2023 British Grand Prix at Silverstone. Max Verstappen (VER, Red Bull Racing) ran stint 1 on MEDIUM tyres from lap 1 to lap 33 (33 laps, tyres reached 33 laps of age).

STINT: 2023 British Grand Prix at Silverstone. Max Verstappen (VER, Red Bull Racing) ran stint 2 on SOFT tyres from lap 34 to lap 52 (19 laps, tyres reached 22 laps of age).

STINT: 2023 British Grand Prix at Silverstone. Pierre Gasly (GAS, Alpine) ran stint 1 on MEDIUM tyres from lap 1 to lap 31 (31 laps, tyres reached 31 laps of age).



In [6]:
#Tier-2: Race Summary chunks 
# One chunk oer driver per race -> get the full race story 

def create_race_summary_chunks(laps_data, session):
    chunks = []
    race_name = session.event['EventName']
    race_year = session.event['EventDate'].year
    circuit   = session.event['Location']
    total_laps = int(laps_data['LapNumber'].max())

    # session.data has official race results
    results = session.results

    for driver in laps_data['Driver'].unique():
        driver_laps = laps_data[laps_data['Driver'] == driver]
        full_name = DRIVER_NAMES.get(driver, driver)
        team = driver_laps['Team'].iloc[0] if len(driver_laps) > 0 else 'Unknown'

        #drivers result
        driver_result = results[results['Abbreviation'] == driver]
        if len(driver_result) == 0:
            continue
        driver_result = driver_result.iloc[0]

        grid_pos = int(driver_result['GridPosition']) if not pd.isna(driver_result['GridPosition']) else 0
        finish_pos = int(driver_result['Position']) if not pd.isna(driver_result['Position']) else 0
        status = driver_result['Status'] if not pd.isna(driver_result['Status']) else "Unknown"

        # Strategy --- compounds ussed in order
        stints = []
        for stint_num in sorted(driver_laps['Stint'].dropna().unique()):
            stint_laps = driver_laps[driver_laps['Stint'] == stint_num]
            compound   = stint_laps['Compound'].iloc[0]
            stints.append(str(compound))

        strategy_str = " → ".join(stints) if stints else "unknown"
        num_stops    = max(0, len(stints) - 1)

        # Position change description
        if grid_pos > 0 and finish_pos > 0:
            change = grid_pos - finish_pos
            if change > 0:
                movement = f"gained {change} positions"
            elif change < 0:
                movement = f"lost {abs(change)} positions"
            else:
                movement = "held position"
        else:
            movement = "position change unknown"

        text = (
            f"RACE SUMMARY: {race_year} {race_name} at {circuit}. "
            f"{full_name} ({driver}, {team}) started P{grid_pos} and "
            f"finished P{finish_pos} ({movement}). "
            f"Strategy: {strategy_str} ({num_stops} stop{'s' if num_stops != 1 else ''}). "
            f"Race status: {status}."
        )
        metadata = {
            'chunk_type'  : 'race_summary',
            'race'        : f"{race_year} {race_name}",
            'circuit'     : circuit,
            'year'        : race_year,
            'driver'      : driver,
            'team'        : str(team),
            'grid_pos'    : grid_pos,
            'finish_pos'  : finish_pos,
            'num_stops'   : num_stops,
            'strategy'    : strategy_str,
        }
        chunks.append({'text': text, 'metadata': metadata})

    return chunks


summary_chunks = create_race_summary_chunks(laps, session)
print(f"Race summary chunks created: {len(summary_chunks)}")
print()
print("Example summaries (first 3):")
for c in summary_chunks[:3]:
    print(c['text'])
    print()

        


Race summary chunks created: 20

Example summaries (first 3):
RACE SUMMARY: 2023 British Grand Prix at Silverstone. Max Verstappen (VER, Red Bull Racing) started P1 and finished P1 (held position). Strategy: MEDIUM → SOFT (1 stop). Race status: Finished.

RACE SUMMARY: 2023 British Grand Prix at Silverstone. Pierre Gasly (GAS, Alpine) started P10 and finished P18 (lost 8 positions). Strategy: MEDIUM → SOFT (1 stop). Race status: Retired.

RACE SUMMARY: 2023 British Grand Prix at Silverstone. Sergio Perez (PER, Red Bull Racing) started P15 and finished P6 (gained 9 positions). Strategy: MEDIUM → SOFT (1 stop). Race status: Finished.



In [8]:
# Check Hamilton's summary specifically
ham_summary = [c for c in summary_chunks if c['metadata']['driver'] == 'HAM']
print(ham_summary[0]['text'])

RACE SUMMARY: 2023 British Grand Prix at Silverstone. Lewis Hamilton (HAM, Mercedes) started P7 and finished P3 (gained 4 positions). Strategy: MEDIUM → SOFT (1 stop). Race status: Finished.


In [10]:
#Combining full 2-tier chunking for one race 
def create_all_chunks(laps_data, session):
    """ 2-tier chunking pipeline for one race, combining tier 1 and 2 into
    single chunk list"""
    all_chunks = []
    all_chunks.extend(create_pitstop_chunks(laps_data, session))
    all_chunks.extend(create_sc_period_chunks(laps_data, session))
    all_chunks.extend(create_stint_chunks(laps_data, session))
    all_chunks.extend(create_race_summary_chunks(laps_data, session))

    return all_chunks

all_chunks = create_all_chunks(laps, session)
print(f" Total chunks for British GP 2023: {len(all_chunks)}" )
print()

#Breaking down by type 
from collections import Counter
type_counts = Counter(c['metadata']['chunk_type'] for c in all_chunks)
print("Breakdown by chunk type:")
for chunk_type, count in type_counts.items():
    print(f" {chunk_type}: {count}")

    

 Total chunks for British GP 2023: 91

Breakdown by chunk type:
 pit_stop: 26
 sc_period: 1
 stint: 44
 race_summary: 20


In [20]:
#Building a multirace pipeline 
#looping through every race in a season

import time

def process_season(year):
    """
    Process every race in a season into chunks 
    
    Paramters
    year : int
    Returns :
        All chunks from all race of that season
    """

    season_chunks = []

    #Season Schedule
    schedule = fastf1.get_event_schedule(year)

    #Filtering only the races on the race days (exlcuding test sessions)
    races = schedule[schedule['RoundNumber'] > 0]
    print(f"Processing {year} season - {len(races)} races") 
    print("==" * 50)

    for _, event in races.iterrows():
        race_name = event['EventName']
        round_num = event['RoundNumber']

        try:
            #Loading race session
            session = fastf1.get_session(year, round_num, 'R')
            session.load(telemetry = False, weather = False, messages = False)
            laps = session.laps

            #No data if race didn't happen
            if len(laps) == 0:
                print(f"Round {round_num}: {race_name} - NO Data, skipped")

            #Running 2-tier chunking
            race_chunks = create_all_chunks(laps, session)
            season_chunks.extend(race_chunks)

            print(f"Round {round_num}: {race_name} - {len(race_chunks)} chunks")

        except Exception as e:
            #if one fails, log it and continue
            print(f"Round {round_num}: {race_name} - Error: {str(e)[:50]}" )
            continue 

    print("==" * 50)
    print(f"{year} completee season: {len(season_chunks)} total chunks")
    return season_chunks

season_2023 = process_season(2023)



Processing 2023 season - 22 races


core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
logger      WARNING 	Failed to load timing data!
core        WARNING 	Failed to add first lap time from Ergast for drivers: ['1', '16', '11', '55', '44', '63', '14', '77', '18', '4', '31', '23', '2', '27', '22', '81', '24', '20', '10', '21']
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '14', '55', '44', '18', '63', '77', '10', '23', '22', '2', '20', '21', '27', '24', '4', '31', '16', '81']


Round 1: Bahrain Grand Prix - Error: The data you are trying to access has not been loa


core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 11 completed the race distance 00:00.035000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['11', '1', '14', '63', '44', '55', '16', '31', '10', '20', '22', '27', '24', '21', '81', '2', '4', '77', '23', '18']
core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	

Round 2: Saudi Arabian Grand Prix - 90 chunks


...

Round 20: São Paulo Grand Prix - Error: The data you are trying to access has not been loa


Request for URL https://raw.githubusercontent.com/theOehrly/f1schedule/master/schedule_2023.json failed; using cached response
Traceback (most recent call last):
  File "/Users/munizasiddiqui/easyf1/venv/lib/python3.11/site-packages/urllib3/connection.py", line 204, in _new_conn
    sock = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/munizasiddiqui/easyf1/venv/lib/python3.11/site-packages/urllib3/util/connection.py", line 60, in create_connection
    for res in socket.getaddrinfo(host, port, family, socket.SOCK_STREAM):
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/munizasiddiqui/.pyenv/versions/3.11.9/lib/python3.11/socket.py", line 962, in getaddrinfo
    for res in _socket.getaddrinfo(host, port, family, type, proto, flags):
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
socket.gaierror: [Errno 8] nodename nor servname provided, or not known

The above exception was the di

Round 21: Las Vegas Grand Prix - Error: The data you are trying to access has not been loa


Request for URL https://raw.githubusercontent.com/theOehrly/f1schedule/master/schedule_2023.json failed; using cached response
Traceback (most recent call last):
  File "/Users/munizasiddiqui/easyf1/venv/lib/python3.11/site-packages/urllib3/connection.py", line 204, in _new_conn
    sock = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/munizasiddiqui/easyf1/venv/lib/python3.11/site-packages/urllib3/util/connection.py", line 60, in create_connection
    for res in socket.getaddrinfo(host, port, family, socket.SOCK_STREAM):
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/munizasiddiqui/.pyenv/versions/3.11.9/lib/python3.11/socket.py", line 962, in getaddrinfo
    for res in _socket.getaddrinfo(host, port, family, type, proto, flags):
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
socket.gaierror: [Errno 8] nodename nor servname provided, or not known

The above exception was the di

Round 22: Abu Dhabi Grand Prix - Error: The data you are trying to access has not been loa
2023 completee season: 930 total chunks


In [13]:
#Downloading and caching one season 2023
#No chunking at this stage 
import fastf1
import time

def download_season(year):
    """
    Download and cache every raace in the season but no chunking at this stage

    """
    schedule = fastf1.get_event_schedule(year)
    races = schedule[schedule['RoundNumber'] > 0]

    successful = []
    failed = []

    print(f"Processing {year} season - {len(races)} races") 
    print("==" * 50)
    
    for _, event in races.iterrows():
        race_name = event['EventName']
        round_num = event['RoundNumber']
    
        try:
            #Loading race session
            session = fastf1.get_session(year, round_num, 'R')
            session.load(telemetry = False, weather = False, messages = False)

            #verify we actually got lap data 
            if len(session.laps) > 0:
                successful.append(round_num)
                print(f"Round {round_num}: {race_name} -- Cached{len(session.laps)} laps)")

            else:
                failed.append(round_num)
                print(f"Round {round_num}: {race_name} -- No lap data")

        except Exception as e:
            failed.append(round_num)
            print(f"Round {round_num}: {race_name} -- Fail{str(e)[:40]} ")

        time.sleep(0.5)

    print ("==" * 50)
    print(f"Successful: {len(successful)} races")
    print(f"Failed: {len(failed)} races --> rounds {failed}")
    print()
    print("If any failed, just re-run this cell — cached races are")
    print("skipped instantly and only missing ones re-download.")
    
    return successful, failed

success_2023, failed_2023 = download_season(2023)




    



core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


Processing 2023 season - 22 races


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
logger      WARNING 	Failed to load timing data!
core        WARNING 	Failed to add first lap time from Ergast for drivers: ['1', '16', '11', '55', '44', '63', '14', '77', '18', '4', '31', '23', '2', '27', '22', '81', '24', '20', '10', '21']
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '14', '55', '44', '18', '63', '77', '10', '23', '22', '2', '20', '21', '27', '24', '4', '31', '16', '81']


Round 1: Bahrain Grand Prix -- FailThe data you are trying to access has no 


core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 11 completed the race distance 00:00.035000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['11', '1', '14', '63', '44', '55', '16', '31', '10', '20', '22', '27', '24', '21', '81', '2', '4', '77', '23', '18']


...

Round 20: São Paulo Grand Prix -- Cached1108 laps)


core           INFO 	Loading data for Las Vegas Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 1 completed the race distance 00:00.001000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['1', '16', '11', '31', '18', '55', '44', '63', '14', '81', '10', '23', '20', '3', '24', '2', '77', '22', '27', '4']


Round 21: Las Vegas Grand Prix -- Cached946 laps)


core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
logger      WARNING 	Failed to load timing data!
core        WARNING 	Failed to add first lap time from Ergast for drivers: ['1', '16', '81', '4', '63', '22', '14', '10', '44', '11', '31', '18', '27', '55', '3', '77', '23', '24', '20', '2']
core           INFO 	Finished loading data for 20 drivers: ['1', '16', '63', '11', '4', '81', '14', '22', '44', '18', '3', '31', '10', '23', '27', '2', '24', '55', '77', '20']


Round 22: Abu Dhabi Grand Prix -- FailThe data you are trying to access has no 
Successful: 16 races
Failed: 6 races --> rounds [1, 6, 7, 13, 19, 22]

If any failed, just re-run this cell — cached races are
skipped instantly and only missing ones re-download.


In [ ]:
#PHASE 2: AUTO-DETECT CACHED RACES + CHUNK
def chunk_available_races(year, max_round=22):
    """
    Tries every round, chunks the ones that load from cache,
    skips the ones that fail. Fully self-contained.
    """
    season_chunks = []
    succeeded = []
    
    print(f"Chunking available {year} races from cache")
    print("="*60)
    
    for round_num in range(1, max_round + 1):
        try:
            session = fastf1.get_session(year, round_num, 'R')
            session.load(telemetry=False, weather=False, messages=False)
            laps = session.laps
            
            if len(laps) == 0:
                print(f"  Round {round_num}: no data, skipped")
                continue
            
            race_chunks = create_all_chunks(laps, session)
            season_chunks.extend(race_chunks)
            succeeded.append(round_num)
            print(f"  Round {round_num}: {len(race_chunks)} chunks ✅")
        
        except Exception as e:
            print(f"  Round {round_num}: skipped (not cached)")
            continue
    
    print("="*60)
    print(f"{year}: {len(season_chunks)} chunks from {len(succeeded)} races")
    return season_chunks, succeeded


chunks_2023, rounds_2023 = chunk_available_races(2023)

Chunking available 2023 races from cache


core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
logger      WARNING 	Failed to load timing data!
core        WARNING 	Failed to add first lap time from Ergast for drivers: ['1', '16', '11', '55', '44', '63', '14', '77', '18', '4', '31', '23', '2', '27', '22', '81', '24', '20', '10', '21']
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '14', '55', '44', '18', '63', '77', '10', '23', '22', '2', '20', '21', '27', '24', '4', '31', '16', '81']
core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.8.3]
req        

  Round 1: skipped (not cached)


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 11 completed the race distance 00:00.035000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['11', '1', '14', '63', '44', '55', '16', '31', '10', '20', '22', '27', '24', '21', '81', '2', '4', '77', '23', '18']
core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info


  Round 2: 90 chunks ✅


...

  Round 20: 176 chunks ✅


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 1 completed the race distance 00:00.001000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['1', '16', '11', '31', '18', '55', '44', '63', '14', '81', '10', '23', '20', '3', '24', '2', '77', '22', '27', '4']
core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


  Round 21: 104 chunks ✅


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
logger      WARNING 	Failed to load timing data!
core        WARNING 	Failed to add first lap time from Ergast for drivers: ['1', '16', '81', '4', '63', '22', '14', '10', '44', '11', '31', '18', '27', '55', '3', '77', '23', '24', '20', '2']
core           INFO 	Finished loading data for 20 drivers: ['1', '16', '63', '11', '4', '81', '14', '22', '44', '18', '3', '31', '10', '23', '27', '2', '24', '55', '77', '20']


  Round 22: skipped (not cached)
2023: 1885 chunks from 16 races


In [17]:
#Verifying chunk quality 
from collections import Counter

#Breakdown by chunk type 
type_counts = Counter(c['metadata']['race'] for c in chunks_2023)
print("Chunk types across 2023: ")
for ct, count in type_counts.items():
    print(f"{ct}: {count}")


print()
print("races covered")
races = sorted(set(c['metadata']['race'] for c in chunks_2023))
for r in races: 
    print (f"{r}")

print()
print("Sample race chunk from a non_British race:")
summaries = [c for c in chunks_2023
             if c ['metadata']['chunk_type'] == 'race_summary'
             and 'British' not in c ['metadata']['race']]

print(summaries[0]['text'])



Chunk types across 2023: 
2023 Saudi Arabian Grand Prix: 90
2023 Australian Grand Prix: 174
2023 Azerbaijan Grand Prix: 88
2023 Miami Grand Prix: 80
2023 Canadian Grand Prix: 109
2023 Austrian Grand Prix: 168
2023 British Grand Prix: 91
2023 Hungarian Grand Prix: 115
2023 Belgian Grand Prix: 115
2023 Italian Grand Prix: 89
2023 Singapore Grand Prix: 90
2023 Japanese Grand Prix: 133
2023 Qatar Grand Prix: 148
2023 United States Grand Prix: 115
2023 São Paulo Grand Prix: 176
2023 Las Vegas Grand Prix: 104

races covered
2023 Australian Grand Prix
2023 Austrian Grand Prix
2023 Azerbaijan Grand Prix
2023 Belgian Grand Prix
2023 British Grand Prix
2023 Canadian Grand Prix
2023 Hungarian Grand Prix
2023 Italian Grand Prix
2023 Japanese Grand Prix
2023 Las Vegas Grand Prix
2023 Miami Grand Prix
2023 Qatar Grand Prix
2023 Saudi Arabian Grand Prix
2023 Singapore Grand Prix
2023 São Paulo Grand Prix
2023 United States Grand Prix

Sample race chunk from a non_British race:
RACE SUMMARY: 2023 Saud

In [20]:
#Saving 2023 chunks to Parquet
import pandas as pd 
import os 

#converting chunks (list of dicts) into a DataFrame
def chunks_to_dataframe(chunks):
    rows = []
    for c in chunks:
        row = {'text': c['text']}
        row.update(c['metadata'])
        rows.append(row)
    return pd.DataFrame(rows)

df_2023 = chunks_to_dataframe(chunks_2023)

#data folder exosts 
os.makedirs('/Users/munizasiddiqui/easyf1/data', exist_ok=True)

#saving to Parquet
parquet_path = '/Users/munizasiddiqui/easyf1/data/chunks_2023.parquet'

print(f"Saved 2023 chunks to Parquet")
print(f"Rows: {len(df_2023)}")
print(f"Columns:{list(df_2023.columns)}")
print(f"File: {parquet_path}")

Saved 2023 chunks to Parquet
Rows: 1885
Columns:['text', 'chunk_type', 'race', 'circuit', 'year', 'lap', 'driver', 'team', 'compound_from', 'compound_to', 'tyre_age', 'position', 'sc_active', 'vsc_active', 'sc_type', 'duration', 'compound', 'stint', 'stint_length', 'grid_pos', 'finish_pos', 'num_stops', 'strategy']
File: /Users/munizasiddiqui/easyf1/data/chunks_2023.parquet


In [25]:
#2022 Phase 1 download
success_2022, failed_2022 = download_season(2022)

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


Processing 2022 season - 22 races


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
logger      WARNING 	Failed to load timing data!
core        WARNING 	Failed to add first lap time from Ergast for drivers: ['16', '1', '55', '44', '20', '11', '63', '10', '14', '31', '23', '22', '47', '77', '27', '4', '18', '6', '24', '3']
core           INFO 	Finished loading data for 20 drivers: ['16', '55', '44', '63', '20', '77', '31', '22', '14', '24', '47', '18', '23', '3', '4', '6', '27', '11', '1', '10']


Round 1: Bahrain Grand Prix -- FailThe data you are trying to access has no 


core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	No lap data for driver 22
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 22)
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 47)
core           INFO 	Finished loading data for 20 drivers: ['1', '16', '55', '11', '63', '31', '4', '10', '20', '44', '24', '27', '18', '23', '77', '14', '3', '6', '22', '47']


...

Round 20: Mexico City Grand Prix -- FailThe data you are trying to access has no 


core           INFO 	Loading data for São Paulo Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core           INFO 	Finished loading data for 20 drivers: ['63', '44', '55', '16', '14', '1', '11', '31', '77', '18', '5', '24', '47', '10', '23', '6', '22', '4', '20', '3']


Round 21: São Paulo Grand Prix -- Cached1259 laps)


core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core           INFO 	Finished loading data for 20 drivers: ['1', '16', '11', '55', '63', '4', '31', '18', '3', '5', '22', '24', '23', '10', '77', '47', '20', '44', '6', '14']


Round 22: Abu Dhabi Grand Prix -- Cached1117 laps)
Successful: 19 races
Failed: 3 races --> rounds [1, 15, 20]

If any failed, just re-run this cell — cached races are
skipped instantly and only missing ones re-download.


In [26]:
#Phase - 2: Chunks
chunks_2022, rounds_2022 = chunk_available_races(2022)

Chunking available 2022 races from cache


core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
logger      WARNING 	Failed to load timing data!
core        WARNING 	Failed to add first lap time from Ergast for drivers: ['16', '1', '55', '44', '20', '11', '63', '10', '14', '31', '23', '22', '47', '77', '27', '4', '18', '6', '24', '3']
core           INFO 	Finished loading data for 20 drivers: ['16', '55', '44', '63', '20', '77', '31', '22', '14', '24', '47', '18', '23', '3', '4', '6', '27', '11', '1', '10']
core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.8.3]
req          

  Round 1: skipped (not cached)


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	No lap data for driver 22
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 22)
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 47)
core           INFO 	Finished loading data for 20 drivers: ['1', '16', '55', '11', '63', '31', '4', '10', '20', '44', '24', '27', '18', '23', '77', '14', '3', '6', '22', '47']
core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


  Round 2: 78 chunks ✅


...

  Round 20: skipped (not cached)


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core           INFO 	Finished loading data for 20 drivers: ['63', '44', '55', '16', '14', '1', '11', '31', '77', '18', '5', '24', '47', '10', '23', '6', '22', '4', '20', '3']
core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


  Round 21: 130 chunks ✅


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core           INFO 	Finished loading data for 20 drivers: ['1', '16', '11', '55', '63', '4', '31', '18', '3', '5', '22', '24', '23', '10', '77', '47', '20', '44', '6', '14']


  Round 22: 105 chunks ✅
2022: 2119 chunks from 19 races


In [31]:
df_2022 = chunks_to_dataframe(chunks_2022)
df_2022.to_parquet('/Users/munizasiddiqui/easyf1/data/chunks_2022.parquet', index=False)
print(f"Saved 2022: {len(df_2022)} chunks")

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.

In [ ]:
import pickle

# Save chunks_2022 to a pickle file (no pyarrow needed)
with open('/Users/munizasiddiqui/easyf1/data/chunks_2022_backup.pkl', 'wb') as f:
    pickle.dump(chunks_2022, f)

print(f"2022 backed up to pickle: {len(chunks_2022)} chunks ")

2022 backed up to pickle: 2119 chunks ✅


In [ ]:
#reload from pickle, save to Parquet
import pickle
import pandas as pd
import os

# Helper to flatten chunks into a DataFrame
def chunks_to_dataframe(chunks):
    rows = []
    for c in chunks:
        row = {'text': c['text']}
        row.update(c['metadata'])
        rows.append(row)
    return pd.DataFrame(rows)

# Reload 2022 from pickle
with open('/Users/munizasiddiqui/easyf1/data/chunks_2022_backup.pkl', 'rb') as f:
    chunks_2022 = pickle.load(f)

print(f"Reloaded 2022: {len(chunks_2022)} chunks")

# Save 2022 to Parquet (pyarrow now available)
df_2022 = chunks_to_dataframe(chunks_2022)
df_2022.to_parquet('/Users/munizasiddiqui/easyf1/data/chunks_2022.parquet', index=False)
print(f"Saved 2022 to Parquet: {len(df_2022)} chunks")

# Verify 2023 Parquet exists (check if it saved earlier)
path_2023 = '/Users/munizasiddiqui/easyf1/data/chunks_2023.parquet'
print(f"2023 Parquet exists: {os.path.exists(path_2023)}")

Reloaded 2022: 2119 chunks
Saved 2022 to Parquet: 2119 chunks ✅
2023 Parquet exists: False


In [ ]:
needed = ['DRIVER_NAMES', 'create_all_chunks', 'chunk_available_races', 'chunks_to_dataframe', 'fastf1']
for name in needed:
    print(f"{name}: {'Done' if name in dir() else 'MISSING'}")

DRIVER_NAMES: ❌ MISSING
create_all_chunks: ❌ MISSING
chunk_available_races: ❌ MISSING
chunks_to_dataframe: ✅
fastf1: ❌ MISSING


In [4]:
# Rebuilding after the restart
import fastf1
import pandas as pd
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

fastf1.Cache.enable_cache('/Users/munizasiddiqui/easyf1/cache')

DRIVER_NAMES = {
    'HAM': 'Lewis Hamilton', 'VER': 'Max Verstappen', 'LEC': 'Charles Leclerc',
    'PER': 'Sergio Perez', 'RUS': 'George Russell', 'SAI': 'Carlos Sainz',
    'NOR': 'Lando Norris', 'GAS': 'Pierre Gasly', 'OCO': 'Esteban Ocon',
    'ALO': 'Fernando Alonso', 'STR': 'Lance Stroll', 'BOT': 'Valtteri Bottas',
    'ZHO': 'Zhou Guanyu', 'MAG': 'Kevin Magnussen', 'HUL': 'Nico Hulkenberg',
    'TSU': 'Yuki Tsunoda', 'DEV': 'Nyck de Vries', 'ALB': 'Alexander Albon',
    'SAR': 'Logan Sargeant', 'PIA': 'Oscar Piastri', 'RIC': 'Daniel Ricciardo',
    'LAW': 'Liam Lawson', 'MSC': 'Mick Schumacher', 'VET': 'Sebastian Vettel',
    'LAT': 'Nicholas Latifi',
}

def create_pitstop_chunks(laps_data, session):
    chunks = []
    race_name = session.event['EventName']; race_year = session.event['EventDate'].year
    circuit = session.event['Location']; total_laps = int(laps_data['LapNumber'].max())
    pit_laps = laps_data[laps_data['PitInTime'].notna()].copy()
    for _, lap in pit_laps.iterrows():
        driver = lap['Driver']; full_name = DRIVER_NAMES.get(driver, driver)
        team = lap['Team']; lap_num = int(lap['LapNumber']); compound = lap['Compound']
        tyre_age = int(lap['TyreLife']) if not pd.isna(lap['TyreLife']) else 0
        position = int(lap['Position']) if not pd.isna(lap['Position']) else 0
        stint = int(lap['Stint']) if not pd.isna(lap['Stint']) else 0
        driver_laps = laps_data[laps_data['Driver'] == driver].sort_values('LapNumber')
        next_laps = driver_laps[driver_laps['LapNumber'] > lap_num]
        new_compound = next_laps.iloc[0]['Compound'] if len(next_laps) > 0 else "unknown"
        track_status = str(lap['TrackStatus']); sc_active = '4' in track_status; vsc_active = '6' in track_status
        condition = "under a Safety Car" if sc_active else "under a Virtual Safety Car" if vsc_active else "under green flag conditions"
        text = (f"PIT STOP: Lap {lap_num} of {total_laps}, {race_year} {race_name} at {circuit}. "
                f"{full_name} ({driver}) racing for {team} made a pit stop from position {position}. "
                f"Switched from {tyre_age}-lap-old {compound} tyres to {new_compound} tyres, {condition}. This ended stint {stint}.")
        metadata = {'chunk_type': 'pit_stop', 'race': f"{race_year} {race_name}", 'circuit': circuit,
                    'year': race_year, 'lap': lap_num, 'driver': driver, 'team': str(team),
                    'compound_from': str(compound), 'compound_to': str(new_compound), 'tyre_age': tyre_age,
                    'position': position, 'sc_active': sc_active, 'vsc_active': vsc_active}
        chunks.append({'text': text, 'metadata': metadata})
    return chunks

def create_sc_period_chunks(laps_data, session):
    chunks = []
    race_name = session.event['EventName']; race_year = session.event['EventDate'].year
    circuit = session.event['Location']; total_laps = int(laps_data['LapNumber'].max())
    lap_status = (laps_data[['LapNumber', 'TrackStatus']].drop_duplicates(subset='LapNumber').sort_values('LapNumber'))
    events = []; current = None
    for _, row in lap_status.iterrows():
        lap = int(row['LapNumber']); status = str(row['TrackStatus'])
        is_sc = '4' in status; is_vsc = '6' in status
        if (is_sc or is_vsc) and current is None:
            current = {'type': 'Safety Car' if is_sc else 'Virtual Safety Car', 'start': lap}
        elif not (is_sc or is_vsc) and current is not None:
            current['end'] = lap - 1; events.append(current); current = None
    if current is not None:
        current['end'] = int(lap_status['LapNumber'].max()); events.append(current)
    for e in events:
        duration = e['end'] - e['start'] + 1
        text = (f"{e['type'].upper()} PERIOD: {race_year} {race_name} at {circuit}. "
                f"A {e['type']} was deployed on lap {e['start']} and lasted until lap {e['end']} "
                f"({duration} laps total, out of {total_laps}). This created a strategic pit window "
                f"where teams could change tyres at reduced time cost.")
        metadata = {'chunk_type': 'sc_period', 'race': f"{race_year} {race_name}", 'circuit': circuit,
                    'year': race_year, 'lap': e['start'], 'sc_type': e['type'], 'duration': duration}
        chunks.append({'text': text, 'metadata': metadata})
    return chunks

def create_stint_chunks(laps_data, session):
    chunks = []
    race_name = session.event['EventName']; race_year = session.event['EventDate'].year
    circuit = session.event['Location']; total_laps = int(laps_data['LapNumber'].max())
    for driver in laps_data['Driver'].unique():
        driver_laps = laps_data[laps_data['Driver'] == driver]
        full_name = DRIVER_NAMES.get(driver, driver)
        team = driver_laps['Team'].iloc[0] if len(driver_laps) > 0 else "Unknown"
        for stint_num in sorted(driver_laps['Stint'].dropna().unique()):
            stint_laps = driver_laps[driver_laps['Stint'] == stint_num]
            if len(stint_laps) == 0: continue
            compound = stint_laps['Compound'].iloc[0]
            first_lap = int(stint_laps['LapNumber'].min()); last_lap = int(stint_laps['LapNumber'].max())
            stint_len = last_lap - first_lap + 1
            max_age = int(stint_laps['TyreLife'].max()) if not pd.isna(stint_laps['TyreLife'].max()) else 0
            text = (f"STINT: {race_year} {race_name} at {circuit}. {full_name} ({driver}, {team}) "
                    f"ran stint {int(stint_num)} on {compound} tyres from lap {first_lap} to lap {last_lap} "
                    f"({stint_len} laps, tyres reached {max_age} laps of age).")
            metadata = {'chunk_type': 'stint', 'race': f"{race_year} {race_name}", 'circuit': circuit,
                        'year': race_year, 'lap': first_lap, 'driver': driver, 'team': str(team),
                        'compound': str(compound), 'stint': int(stint_num), 'stint_length': stint_len}
            chunks.append({'text': text, 'metadata': metadata})
    return chunks

def create_race_summary_chunks(laps_data, session):
    chunks = []
    race_name = session.event['EventName']; race_year = session.event['EventDate'].year
    circuit = session.event['Location']; results = session.results
    for driver in laps_data['Driver'].unique():
        driver_laps = laps_data[laps_data['Driver'] == driver]
        full_name = DRIVER_NAMES.get(driver, driver)
        team = driver_laps['Team'].iloc[0] if len(driver_laps) > 0 else "Unknown"
        driver_result = results[results['Abbreviation'] == driver]
        if len(driver_result) == 0: continue
        driver_result = driver_result.iloc[0]
        grid_pos = int(driver_result['GridPosition']) if not pd.isna(driver_result['GridPosition']) else 0
        finish_pos = int(driver_result['Position']) if not pd.isna(driver_result['Position']) else 0
        status = driver_result['Status'] if not pd.isna(driver_result['Status']) else "Unknown"
        stints = []
        for stint_num in sorted(driver_laps['Stint'].dropna().unique()):
            stint_laps = driver_laps[driver_laps['Stint'] == stint_num]
            stints.append(str(stint_laps['Compound'].iloc[0]))
        strategy_str = " → ".join(stints) if stints else "unknown"
        num_stops = max(0, len(stints) - 1)
        if grid_pos > 0 and finish_pos > 0:
            change = grid_pos - finish_pos
            movement = f"gained {change} positions" if change > 0 else f"lost {abs(change)} positions" if change < 0 else "held position"
        else:
            movement = "position change unknown"
        text = (f"RACE SUMMARY: {race_year} {race_name} at {circuit}. {full_name} ({driver}, {team}) "
                f"started P{grid_pos} and finished P{finish_pos} ({movement}). "
                f"Strategy: {strategy_str} ({num_stops} stop{'s' if num_stops != 1 else ''}). Race status: {status}.")
        metadata = {'chunk_type': 'race_summary', 'race': f"{race_year} {race_name}", 'circuit': circuit,
                    'year': race_year, 'driver': driver, 'team': str(team), 'grid_pos': grid_pos,
                    'finish_pos': finish_pos, 'num_stops': num_stops, 'strategy': strategy_str}
        chunks.append({'text': text, 'metadata': metadata})
    return chunks

def create_all_chunks(laps_data, session):
    all_chunks = []
    all_chunks.extend(create_pitstop_chunks(laps_data, session))
    all_chunks.extend(create_sc_period_chunks(laps_data, session))
    all_chunks.extend(create_stint_chunks(laps_data, session))
    all_chunks.extend(create_race_summary_chunks(laps_data, session))
    return all_chunks

def chunk_available_races(year, max_round=22):
    season_chunks = []; succeeded = []
    print(f"Chunking available {year} races from cache"); print("="*60)
    for round_num in range(1, max_round + 1):
        try:
            session = fastf1.get_session(year, round_num, 'R')
            session.load(telemetry=False, weather=False, messages=False)
            laps = session.laps
            if len(laps) == 0:
                print(f"  Round {round_num}: no data, skipped"); continue
            race_chunks = create_all_chunks(laps, session)
            season_chunks.extend(race_chunks); succeeded.append(round_num)
            print(f"  Round {round_num}: {len(race_chunks)} chunks ")
        except Exception as e:
            print(f"  Round {round_num}: skipped (not cached)"); continue
    print("="*60); print(f"{year}: {len(season_chunks)} chunks from {len(succeeded)} races")
    return season_chunks, succeeded

def chunks_to_dataframe(chunks):
    rows = []
    for c in chunks:
        row = {'text': c['text']}; row.update(c['metadata']); rows.append(row)
    return pd.DataFrame(rows)

print("Everything rebuilt ")

Everything rebuilt 


In [ ]:
chunks_2023, rounds_2023 = chunk_available_races(2023)

# Pickle backup FIRST (lesson learned)
with open('/Users/munizasiddiqui/easyf1/data/chunks_2023_backup.pkl', 'wb') as f:
    pickle.dump(chunks_2023, f)

# Then Parquet
df_2023 = chunks_to_dataframe(chunks_2023)
df_2023.to_parquet('/Users/munizasiddiqui/easyf1/data/chunks_2023.parquet', index=False)
print(f"Saved 2023: {len(df_2023)} chunks ")

Chunking available 2023 races from cache


core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
logger      WARNING 	Failed to load timing data!
core        WARNING 	Failed to add first lap time from Ergast for drivers: ['1', '16', '11', '55', '44', '63', '14', '77', '18', '4', '31', '23', '2', '27', '22', '81', '24', '20', '10', '21']
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '14', '55', '44', '18', '63', '77', '10', '23', '22', '2', '20', '21', '27', '24', '4', '31', '16', '81']
core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.8.3]
req        

  Round 1: skipped (not cached)


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 11 completed the race distance 00:00.035000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['11', '1', '14', '63', '44', '55', '16', '31', '10', '20', '22', '27', '24', '21', '81', '2', '4', '77', '23', '18']
core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


  Round 2: 90 chunks ✅


...

  Round 20: 176 chunks ✅


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 1 completed the race distance 00:00.001000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['1', '16', '11', '31', '18', '55', '44', '63', '14', '81', '10', '23', '20', '3', '24', '2', '77', '22', '27', '4']
core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


  Round 21: 104 chunks ✅


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
logger      WARNING 	Failed to load timing data!
core        WARNING 	Failed to add first lap time from Ergast for drivers: ['1', '16', '81', '4', '63', '22', '14', '10', '44', '11', '31', '18', '27', '55', '3', '77', '23', '24', '20', '2']
core           INFO 	Finished loading data for 20 drivers: ['1', '16', '63', '11', '4', '81', '14', '22', '44', '18', '3', '31', '10', '23', '27', '2', '24', '55', '77', '20']


  Round 22: skipped (not cached)
2023: 2007 chunks from 17 races
Saved 2023: 2007 chunks ✅


In [7]:
def download_season(year):
    schedule = fastf1.get_event_schedule(year)
    races = schedule[schedule['RoundNumber'] > 0]
    successful = []; failed = []
    print(f"Downloading {year} season — {len(races)} races"); print("="*60)
    for _, event in races.iterrows():
        race_name = event['EventName']; round_num = event['RoundNumber']
        try:
            session = fastf1.get_session(year, round_num, 'R')
            session.load(telemetry=False, weather=False, messages=False)
            if len(session.laps) > 0:
                successful.append(round_num)
                print(f"  Round {round_num}: {race_name} — cached ✅ ({len(session.laps)} laps)")
            else:
                failed.append(round_num)
                print(f"  Round {round_num}: {race_name} — NO LAP DATA ⚠️")
        except Exception as e:
            failed.append(round_num)
            print(f"  Round {round_num}: {race_name} — FAILED: {str(e)[:40]}")
        import time; time.sleep(0.5)
    print("="*60)
    print(f"Successful: {len(successful)} races")
    print(f"Failed: {len(failed)} races → rounds {failed}")
    return successful, failed

In [12]:
#2021 Phase - 1: Download
success_2021, failed_2021 = download_season(2021)

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
logger      WARNING 	Failed to load timing data!
core        WARNING 	Failed to add first lap time from Ergast for drivers: ['33', '44', '16', '77', '10', '4', '3', '14', '18', '55', '7', '99', '31', '5', '22', '63', '6', '47', '11']
core           INFO 	Finished loading data for 20 drivers: ['44', '33', '77', '4', '11', '16', '3', '55', '22', '18', '7', '99', '31', '63', '5', '47', '10', '6', '14', '9']


  Round 1: Bahrain Grand Prix — FAILED: The data you are trying to access has no


core           INFO 	Loading data for Emilia Romagna Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 33 completed the race distance 00:01.003000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['33', '44', '4', '16', '55', '3', '10', '18', '31', '14', '11', '22', '7', '99', '5', '47', '9', '77', '63', '6']


  Round 2: Emilia Romagna Grand Prix — cached ✅ (1127 laps)


...

  Round 20: Qatar Grand Prix — cached ✅ (1114 laps)


core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 44 completed the race distance 00:00.180000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['44', '33', '77', '31', '3', '10', '16', '55', '99', '4', '18', '6', '14', '22', '7', '5', '11', '9', '63', '47']


  Round 21: Saudi Arabian Grand Prix — cached ✅ (845 laps)


core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 9)
core        WARNING 	Driver 33 completed the race distance 00:00.035000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['33', '44', '55', '22', '10', '77', '4', '14', '31', '16', '5', '3', '18', '47', '11', '6', '99', '63', '7', '9']


  Round 22: Abu Dhabi Grand Prix — cached ✅ (1000 laps)
Successful: 18 races
Failed: 4 races → rounds [1, 3, 5, 13]


In [13]:
#2021 Phase - 2: CHUNK + save
chunks_2021, rounds_2021 = chunk_available_races(2021)

# Pickle backup FIRST
with open('/Users/munizasiddiqui/easyf1/data/chunks_2021_backup.pkl', 'wb') as f:
    pickle.dump(chunks_2021, f)

# Then Parquet
df_2021 = chunks_to_dataframe(chunks_2021)
df_2021.to_parquet('/Users/munizasiddiqui/easyf1/data/chunks_2021.parquet', index=False)
print(f"Saved 2021: {len(df_2021)} chunks ✅")

Chunking available 2021 races from cache


core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
logger      WARNING 	Failed to load timing data!
core        WARNING 	Failed to add first lap time from Ergast for drivers: ['33', '44', '16', '77', '10', '4', '3', '14', '18', '55', '7', '99', '31', '5', '22', '63', '6', '47', '11']
core           INFO 	Finished loading data for 20 drivers: ['44', '33', '77', '4', '11', '16', '3', '55', '22', '18', '7', '99', '31', '63', '5', '47', '10', '6', '14', '9']
core           INFO 	Loading data for Emilia Romagna Grand Prix - Race [v3.8.3]
req            INFO 	

  Round 1: skipped (not cached)


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 33 completed the race distance 00:01.003000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['33', '44', '4', '16', '55', '3', '10', '18', '31', '14', '11', '22', '7', '99', '5', '47', '9', '77', '63', '6']


  Round 2: 155 chunks ✅


...

  Round 20: 102 chunks ✅


core        WARNING 	Driver 44 completed the race distance 00:00.180000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['44', '33', '77', '31', '3', '10', '16', '55', '99', '4', '18', '6', '14', '22', '7', '5', '11', '9', '63', '47']
core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


  Round 21: 139 chunks ✅


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 9)
core        WARNING 	Driver 33 completed the race distance 00:00.035000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['33', '44', '55', '22', '10', '77', '4', '14', '31', '16', '5', '3', '18', '47', '11', '6', '99', '63', '7', '9']


  Round 22: 101 chunks ✅
2021: 2078 chunks from 18 races
Saved 2021: 2078 chunks ✅


In [ ]:
#Combining all three season to into one and then embedding to ChromaDB
import pandas as pd 
#Load all three prequet files 
df_2021 = pd.read_parquet('/Users/munizasiddiqui/easyf1/data/chunks_2021.parquet')
df_2022 = pd.read_parquet('/Users/munizasiddiqui/easyf1/data/chunks_2022.parquet')
df_2023 = pd.read_parquet('/Users/munizasiddiqui/easyf1/data/chunks_2023.parquet')

#Combining into one DataFrame
df_all = pd.concat([df_2021, df_2022, df_2023], ignore_index=True)

#Saving comnined file
df_all.to_parquet('/Users/munizasiddiqui/easyf1/data/chunks_all_seasons.parquet', index=False)

print(f"Complete Knowledge base")
print(f"Total_chunks: {len(df_all)}")
print()
print(f"Breakdown by year")
print(df_all['year'].value_counts().sort_index())
print()
print(f"Breakdown by chunk type:")
print(df_all['chunk_type'].value_counts())

Complete Knowledge base
Total_chunks: 6204

Breakdown by year
year
2021    2078
2022    2119
2023    2007
Name: count, dtype: int64

Breakdown by chunk type:
chunk_type
stint           3014
pit_stop        2042
race_summary    1072
sc_period         76
Name: count, dtype: int64


In [15]:
#Embedding to ChromaDB
import chromadb
from fastembed import TextEmbedding

#loading embedding model
embedding_model = TextEmbedding(
    model_name = "BAAI/bge-small-en-v1.5",
    cache_dir= "/Users/munizasiddiqui/easyf1/model_cache"
)
def embed_text(text):
    return  list(embedding_model.embed([text]))[0].tolist()

#Coonecting to chromadb
chroma_client = chromadb.PersistentClient(path='/Users/munizasiddiqui/easyf1/chromadb')

#Creating new collection for multiseason data
try:
    chroma_client.delete_collection("easyf1_all_seasons")
except Exception:
    pass

collection = chroma_client.create_collection(
    name ="easyf1_all_seasons",
    metadata ={"hnsw:space": "cosine" }
)
print("Fresh collection 'easyf1_all_seasons' created")

#embedding and storing in batches
batch_size = 200
total = len(df_all)
print(f"Embedding {total} chunks locally.")
print()

for i in range (0, total, batch_size):
    batch = df_all.iloc[i:i + batch_size]
    
    texts = batch['text'].tolist()
    ids = [f"chunk_{j}" for j in range(i, i + len(batch))]

    #Build metadata - handle nulls from mixed chunk types
    metadatas = []
    for _, row in batch.iterrows():
        meta = {}

        for col in df_all.columns:
            if col == 'text':
                continue
            val = row[col]

            # Skip nulls (different chunk types have different fields)
            if pd.isna(val):
                continue
            # Cast to correct type
            if col in ['year', 'lap', 'tyre_age', 'position', 'duration',
                       'stint', 'stint_length', 'grid_pos', 'finish_pos', 'num_stops']:
                meta[col] = int(val)
            elif col in ['sc_active', 'vsc_active']:
                meta[col] = bool(val)
            else:
                meta[col] = str(val)
        metadatas.append(meta)

    embeddings = [embed_text(t) for t in texts]

    collection.add(
        documents = texts,
        embeddings = embeddings,
        metadatas = metadatas,
        ids = ids
    )
    print(f" {min(i + batch_size, total)} / {total} embedded...")

print()
print(f"all done")
print(f"Total in ChromaDB: {collection.count()}")

        


Fresh collection 'easyf1_all_seasons' created
Embedding 6204 chunks locally.

 200 / 6204 embedded...
 400 / 6204 embedded...
 600 / 6204 embedded...
 800 / 6204 embedded...
 1000 / 6204 embedded...
 1200 / 6204 embedded...
 1400 / 6204 embedded...
 1600 / 6204 embedded...
 1800 / 6204 embedded...
 2000 / 6204 embedded...
 2200 / 6204 embedded...
 2400 / 6204 embedded...
 2600 / 6204 embedded...
 2800 / 6204 embedded...
 3000 / 6204 embedded...
 3200 / 6204 embedded...
 3400 / 6204 embedded...
 3600 / 6204 embedded...
 3800 / 6204 embedded...
 4000 / 6204 embedded...
 4200 / 6204 embedded...
 4400 / 6204 embedded...
 4600 / 6204 embedded...
 4800 / 6204 embedded...
 5000 / 6204 embedded...
 5200 / 6204 embedded...
 5400 / 6204 embedded...
 5600 / 6204 embedded...
 5800 / 6204 embedded...
 6000 / 6204 embedded...
 6200 / 6204 embedded...
 6204 / 6204 embedded...

all done
Total in ChromaDB: 6204


In [17]:
# Testing cross race retrieval
def quick_query(question, n=5, where=None):
    q_emb = embed_text(question)
    params = {"query_embeddings": [q_emb], "n_results": n,
              "include": ['documents', 'metadatas']}
    if where:
        params["where"] = where
    return collection.query(**params)

# Test 1: Cross-season question
print("TEST 1: Who won races from pole across seasons?")
print("="*60)
r = quick_query("driver started P1 pole position and won the race", n=5)
for doc, meta in zip(r['documents'][0], r['metadatas'][0]):
    if meta['chunk_type'] == 'race_summary':
        print(f"  {meta['race']}: {doc[:90]}")
print()

# Test 2: Circuit-specific across seasons
print("TEST 2: Safety car periods at Silverstone (any year)")
print("="*60)
r = quick_query("safety car deployed", n=10,
                where={"circuit": {"$eq": "Silverstone"}})
for doc, meta in zip(r['documents'][0], r['metadatas'][0]):
    if meta['chunk_type'] == 'sc_period':
        print(f"  {doc[:100]}")
print()

# Test 3: Strategy comparison
print("TEST 3: One-stop MEDIUM to HARD strategies")
print("="*60)
r = quick_query("one stop strategy medium to hard tyres", n=5)
for doc, meta in zip(r['documents'][0], r['metadatas'][0]):
    if meta['chunk_type'] == 'race_summary':
        print(f"  {meta['race']}: {doc[:90]}")

TEST 1: Who won races from pole across seasons?
  2023 British Grand Prix: RACE SUMMARY: 2023 British Grand Prix at Silverstone. Max Verstappen (VER, Red Bull Racing
  2023 Japanese Grand Prix: RACE SUMMARY: 2023 Japanese Grand Prix at Suzuka. Max Verstappen (VER, Red Bull Racing) st
  2023 Qatar Grand Prix: RACE SUMMARY: 2023 Qatar Grand Prix at Lusail. Max Verstappen (VER, Red Bull Racing) start
  2022 Japanese Grand Prix: RACE SUMMARY: 2022 Japanese Grand Prix at Suzuka. Max Verstappen (VER, Red Bull Racing) st
  2023 Canadian Grand Prix: RACE SUMMARY: 2023 Canadian Grand Prix at Montréal. Max Verstappen (VER, Red Bull Racing) 

TEST 2: Safety car periods at Silverstone (any year)
  SAFETY CAR PERIOD: 2021 British Grand Prix at Silverstone. A Safety Car was deployed on lap 1 and la
  SAFETY CAR PERIOD: 2023 British Grand Prix at Silverstone. A Safety Car was deployed on lap 33 and l
  SAFETY CAR PERIOD: 2022 British Grand Prix at Silverstone. A Safety Car was deployed on lap 38 and 